# Flow Matching

不推荐你读 https://arxiv.org/abs/2210.02747 Flow Matching for Generative Modeling 即使这篇论文是本章的重中之重。没有看错，是不推荐你读，原因是我个人认为这篇文章可读性很低，要求了许多前置知识储备。我个人更推荐从 MIT 6.S184 https://diffusion.csail.mit.edu/2026/index.html Introduction to Flow Matching and Diffusion Models 入门，主讲 Peter Holderrieth 讲解得通俗易懂。需要提醒，FM 的内容数学推导会更硬核，如果你希望掌握本质内容，推荐你拥有多元微积分 (如 Frechet 微分)，拓扑与测度论的基本认识。不过我会尽量说得简单。

# 前置知识

这里是前置知识，推荐了解，但是不完全掌握也不会阻碍重大的可读性。我们后续讲解工程中的训练和推理时完全不会涉及这部分内容。

我们简单交代测度为何物。简而言之，测度就是对于集合大小的衡量函数。当你做积分，实际上无意中指代了对某个测度做积分，比如一维实数空间上一般就是 Lebesgue 测度。对于实数区间 $[a, b)$，Lebesgue 测度中这个区间的大小是 $$m([a, b)) = b - a$$
这很符合直觉。Lebesgue 积分的标准写法通常包含测度项 $dm$，比如在集合 $E$ 上积分，记作 $$\int_E f \, dm \quad \text{或} \quad \int_E f(x) \, d\lambda(x)$$

如果你从未听说过测度，你很有可能从未感受到测度的存在，原因通常是你习惯使用 Riemann 积分，然而测度却是 Lebesgue 积分引入的概念。

所以具体如何积分？理论上我们应该交代 Beppo Levi 定理等等一系列构建 Lebesgue 积分理论的内容，但是我们没那么多时间，况且我们一般只对美好的可积的概率密度函数做积分，他们通常都是 Lebesgue 可积的，并且性质优良。我们快速给出积分方法。

对于一个可以被表示为有限个可测集合的指示函数的线性组合的简单函数 $\phi$  $$\phi(x) = \sum_{i=1}^{n} a_i \chi_{E_i}(x)$$其中 $a_i$ 是非负实数，$E_i$ 是可测集。$\chi_{A}(x)$ 是指示函数，其在集合 $A$ 上值为 $1$，其余地方为 $0$。

关于什么是可测集合，简而言之就是可以被测度描述的集合。这很诡异，因为一般认知里所有集合都理应有个测度大小，实际上不是的。我们指出，某些集合无法被描述大小，比如 Vitali 集合，他们被称为不可测。但是绝大绝大多数集合都是可测的，我们甚至完全遇不到不可测集合。我们继续说。

对于定义在测度空间 $(X, \mathcal{F}, m)$ 上的非负简单函数 $\phi$，其对测度 $m$ 的积分定义为 $$\int_X \phi \, dm = \sum_{i=1}^{n} a_i m(E_i)$$

如果把此处的测度 $m$ 设定为 Lebesgue 测度，实际上这就是常见的积分形式。比如函数 $\chi_{[0,2)}$ 在全空间上的积分就是 $1 \times 2 = 2$。

接下来我们讨论一般函数的积分，但是我们限定只考虑可积函数积分。因为我们将来要做的概率密度函数积分就是可积函数。

对于可积函数 $f$，对于每个 $n \in \mathbb{N}$，将函数 $f$ 值域划分为 $n \cdot 2^n$ 个小区间 $$E_{n,k} = \left\{ x \in X : \frac{k-1}{2^n} \leq f(x) < \frac{k}{2^n} \right\}, \quad k = 1, 2, \dots, n \cdot 2^n$$
以及超出范围的集合 $$F_n = \{ x \in X : f(x) \geq n \}$$

使用指示函数来构造每一层的简单函数 $$f_n(x) = \sum_{k=1}^{n \cdot 2^n} \frac{k-1}{2^n} \chi_{E_{n,k}}(x) + n \chi_{F_n}(x)$$

该序列 $\{f_n\}$ 满足单调性、逐点收敛性以及积分的可计算性。我指 $$\forall x \in X, \quad f_n(x) \leq f_{n+1}(x) \leq f(x), \quad \lim_{n \to \infty} f_n(x) = f(x)$$
并且由于 $f_n$ 是简单函数，其积分定义为测度的有限线性组合 $$\int_X f_n \, dm = \sum_{k=1}^{n \cdot 2^n} \frac{k-1}{2^n} m(E_{n,k}) + n \cdot m(F_n)$$

最终，$f$ 的 Lebesgue 积分定义为这些简单函数积分的极限 $$\int_X f \, dm = \lim_{n \to \infty} \int_X f_n \, dm$$

接近说完了。总之我们极简构建了 Lebesgue 积分和测度理论。我必须阐述这样做的动机。一个概率密度函数的意义在于对于某一区域的积分描述数据分布出现在此的概率，而当空间发生扭曲，我们需要新的测度来描述对于这一区域的积分，这就是测度变换的基本思想。

在接下来对于 Flow Matching 理论的构建中，我们会遇到概率密度函数的迁移，也就是概率密度路径。对于这种路径，我们需要测度变换来描述。我们开始吧。

## 基本逻辑

在 Score-based SDE 中，模型学习的是得分场 $\nabla \log p_t$；而在 FM 中，模型学习的是速度场 $\mathbf{v}_t(\mathbf{x})$。

FM 的基本思路非常直观。我们定义一个随时间变化的概率路径 $p_t$，其将噪声分布 $p_0$ 演化为数据分布 $p_1$。这个路径必然对应一个矢量场 $\mathbf{u}_t$。我们训练一个神经网络 $\mathbf{v}_{\theta}(\mathbf{x}, t)$，直接去回归这个目标矢量场 $$\min_{\theta} \mathbb{E}_{t, p_t(\mathbf{x})} \| \mathbf{v}_{\theta}(\mathbf{x}, t) - \mathbf{u}_t(\mathbf{x}) \|^2$$

这样说可能让人一头雾水。我们接下来详谈。

# 轨迹，矢量场与流

### 通俗解释

我必须指出，Flow Matching 的抽象程度和符号化程度非常高，但是我会尽量使用不失严谨且易懂的语言叙述这一切。现在我们开始。

考虑在数据空间 $\mathbb{R}^d$ 中的真实图片分布 $p_{data}$。我们认为，存在一种变换，能从我们给定的初始分布 $p_{init}$ (一般是高斯分布) 变换到真实分布 $p_{data}$。假设原本有一群点遵循 $p_{init}$ 分布，随后每点受到某种力作用沿着某一轨迹迁移到时间 $t$ 下新的位置，最终新的位置上所有点形成新的分布 $p_{data}$。如果我们能够研究这个迁移轨迹，我们就可以掌握真实的数据分布。

我在此处放入一个视频，演示了这种变化过程，非常形象。你可与看到原本高斯分布的点集合如何演变到另一个分布。资源来自 MIT 6.S184 课程。

<video src="./assets/ShowFM.mp4" controls width="600" autoplay loop muted>
</video>

我们定义轨迹，指的是一个集合 $X$，内容是 $\{{x}_0, \dots, {x}_1\}$，其中 ${x}_t$ 是在时间 $t$ 时，点 $x$ 在多维空间里的位置坐标。

我们定义矢量场 $u$，给出多维空间里每一个位置在每一时刻的速度，其作为一个映射是 $u: [0, 1] \times \mathbb{R}^d \to \mathbb{R}^d$ 接收一个时间步 $t$ 与一个位置 $x$，给出一个速度矢量。换言之，对于某一时间步 $t$ $$u_t({x}_t) = {v}$$ $$\frac{d{x}_t}{dt} = u_t({x}_t)$$

矢量场定义了运动规则。在一个多维空间里，给定起点 $x_0$，给定了向量场 $u$，则轨迹是确定的。下面给出一个矢量场演示图，你可以看到每一点对应一个箭头表示速度方向。

<img src="./assets/ShowField.GIF" width="400" height="300">

接下来我们考虑轨迹的描述，也就是流 (Flow)，用 $\phi$ 表示，是一系列轨迹的集合。每个轨迹都按照矢量场 $u$ 运动。

流作为映射是 $\phi: [0, 1] \times \mathbb{R}^d \to \mathbb{R}^d$ 接收一个时间步 $t$ 与一个初始位置 $x_0$，给出一个末位置 $x_t$。因此流本质是矢量场对于时间的积分。更多的，因为 $${x}_t = \phi_t({x}_0)$$
且刚刚提到 $$\frac{d{x}_t}{dt} = u_t({x}_t)$$
得到流与矢量场直接关系 $$\frac{d\phi_t({x}_0)}{dt} = u_t(\phi_t({x}_0)) \quad (*)$$
下面这张图展示了流的形成。

<img src="./assets/ShowFlow.GIF" width="400" height="300">

流 $\phi$ 对于时间步 $t$ 展开是一个单参数微分同胚族 $\phi_t: \mathbb{R}^d \to \mathbb{R}^d$。解释一下什么是微分同胚 (Diffeomorphism)，简而言之同构就是满射且单射的映射，同胚就是本身连续且逆映射也连续的同构，微分同胚就是 $k$ 阶微分连续 ( $C^k$ 连续) 的同胚，称为$C^k$ 级微分同胚。在 $k$ 为 $0$ 时，微分同胚就是同胚。

一个关于 $(*)$ 的结论是，如果我们定义的矢量场 $u$ 有全局 Lipschitz 连续性 (任意两点之间斜率存在上界)，且 $u_t(x)$ 在 $t \in [0, 1]$ 上关于 $t$ 连续且在空间维度 $\mathbb{R}^d$ 上是 $C^k$ 连续的，那么就确保了解得的 $\phi_t$ 自动是 $C^k$ 级微分同胚。请注意，现在的条件是结论的充分条件，我们暂时不关心等价条件。

我们想说的，研究轨迹 $\phi$ 本质是研究矢量场 $u$，因此我们希望一个神经网络来拟合矢量场。

正式定义概率密度路径 $p: [0, 1] \times \mathbb{R}^d \to \mathbb{R}_{>0}$。其作为一个映射接收一个时间步 $t$，变成某个时间步下的概率密度函数 $p_t$。

所以一个轨迹注定决定一个概率密度路径，因为我们知道每一点的运动轨迹，那么理论上可以得到任意时间步 $t$ 的点分布情况。我们给出计算方法。若已知初始时刻的先验分布为 $p_0$，则在时刻 $t$ 的密度 $p_t$ 由流 $\phi_t$ 对 $p_0$ 做测度变换的 Push-forward 操作得到，记作 $$p_t = [\phi_t]_* p_0$$
这里测度变换的严谨含义是，对于 $\mathbb{R}^d$ 中的任意 Borel 集合 $A$，满足 $$\int_A p_t(x) \mathrm{d}x = \int_{\phi_t^{-1}(A)} p_0(x) \mathrm{d}x$$这保证了在变换过程中，概率总质量始终守恒且积分为 1。

我们耐心解释上面这段数学语言。首先，$\mathbb{R}^d$ 上 Borel 集合就是开集经过可数并或交与补运算得到的集合，几乎任何常见的集合都是 Borel 集合，且 Borel 集合一定 Lebesgue 可测，因此这里不必重点关注。其次的，为什么我们这里要用积分写出概率密度路径中一点 $p_t$？原因正是我们早就提及的，概率密度函数的动机就是考虑某个区域上积分来表示数据分布在此处的概率，因此使用积分书写是合理的。最后的，为什么这个积分是这个形式？实际上很容易理解。请看下面的变换。由于 $\phi_t$ 是个微分同胚，我们可以换种形式重新书写。

定义 $B$ 为起始空间（高斯噪声空间）中的一个 Borel 集，而 $A$ 是它在时刻 $t$ 的像，即 $A = \phi_t(B)$，那么由于 $\phi_t$ 是微分同胚，原本的式子可以等价地写为$$\int_{\phi_t(B)} p_t(x) \mathrm{d}x = \int_B p_0(y) \mathrm{d}y$$
这表示在初始分布 $p_0$ 中处于集合 $B$ 内的所有概率质量，经过流 $\phi_t$ 的演化后，在时刻 $t$ 必然完全落入集合 $\phi_t(B)$ 中。

总之我们可以写出积分形式。更进一步的，我们可以写出解析形式。根据变量代换定理（Change of Variables Formula），该过程在代数上等价于 $$p_t(x) = p_0(\phi_t^{-1}(x)) \det \left[ \nabla_x \phi_t^{-1}(x) \right]$$上式中，$\nabla_x \phi_t^{-1}(x)$ 表示逆映射 $\phi_t^{-1}$ 的 Jacobian 矩阵，其行列式绝对值反映了映射过程中的测度伸缩比例。

上式具体推导我们不做证明，实际上如果熟悉测度论，比想象中会更简单。

因此，只要知道初始分布 $p_0$ 和流的逆映射，就能精确计算任意时刻任意位置的概率密度。这保证了理论的基础。

更直接的，我们可以直接给出矢量场 $u$ 与概率分布 $p_t$ 之间的关系，这意味着我们可以避免 $\phi_t$ 的计算。

若矢量场 $v_t$ 与概率路径 $p_t$ 满足上述 Push-forward 关系，则在分布意义下，它们必须满足一阶偏微分方程，我们称之为连续性方程 $$\frac{\partial p_t(x)}{\partial t} + \nabla \cdot (p_t(x) u_t(x)) = 0$$其中 $\nabla \cdot$ 表示散度算子（Divergence）。该方程在物理上对应质量守恒定律，在测度论中确保了概率测度在流变换过程中的总积分为 1。

换言之，给定$$ x_0 \sim p_{\text{init}}, \quad \frac{\text{d}}{\text{dt}} x_t = u_t(x_t)$$
且其遵循 $$x_t \sim p_t \quad (0 \leq t \leq 1)$$
那么方程成立 $$\frac{d}{dt} p_t(x) = -\text{div}(p_t u_t)(x)$$

上式是最不容易理解的，我们会在附录中详细证明这一点。现在只需要理解这个结论即可。

# 训练与目标函数

假设我们已经构造了一个目标概率路径 $p_t(x)$ 以及能够生成该路径的理想目标矢量场 $u_t(x)$，我们定义损失函数
$$\mathcal{L}_{\text{FM}}(\theta) = \mathbb{E}_{t \sim \mathcal{U}[0,1], x \sim p_t(x)} \|v_t(x; \theta) - u_t(x)\|^2$$
其中 $\mathcal{U}$ 是指均匀分布。这是一个 MSE 距离。其目标是让神经网络定义的矢量场 $v_t(x; \theta)$ 在全时空尺度上逼近目标矢量场 $u_t(x)$。

不过原始的问题是，由于 $p_{data}$ 未知，由所有数据样本聚合而成的目标概率路径 $p_t(x)$ 及其对应的速度场 $u_t(x)$ 均没有解析表达式。你会发现我们之前提到的几乎所有损失函数计算都有这个问题，我们一直在尝试工程上的估计手法。下面为你展示。

### 条件概率路径，条件流与条件矢量场

我们引入条件概率路径（Conditional Probability Path）的概念。其核心思想是，不直接考虑整个数据分布的演化，而是考虑单一数据样本 $x_1$ 的演化路径。

给定特定样本 $z \sim p_{data}$，定义条件路径 $p_t(x|z)$，含义是已知最终在时间步 $t$ 为 $1$ 时粒子集中在 $z$ 附近，在时间步为 $t$ 下粒子出现在 $x$ 的概率。严谨的说法是，对于空间 $\mathbb{R}^d$ 中的一个无限小体积元 $\mathrm{d}x$ 位于坐标 $x$ 处，在已知目标样本为 $z$ 的前提下，粒子在 $t$ 时刻落在该体积元内的概率 $\mathrm{d}P$ 为 $$\mathrm{d}P = p_t(x|z) \mathrm{d}x$$ 更加详细的，$t=0$ 时，$p_0(x|z) = p(x)$，比如简单的先验高斯分布；$t=1$ 时，$p_1(x|z)$ 是一个以 $z$ 为中心的高度集中的分布，比如方差极小的各向同性高斯分布 $\mathcal{N}(x|x_1, \sigma^2 I)$，或者 Dirac 分布 $\delta_{x_1}$。Dirac 分布指的是极其简单的，任意的采样只有一个结果 $z$ 的分布。

这里给一个简单例子，名为最优路径。定义 $p_t(\cdot | z) = \mathcal{N}(\alpha_t z, \beta_t^2 \mathbf{I})$，其中 $\alpha_t = t$ 且 $\beta_t = 1 - t$。在 $t$ 为 $0$，$p_t(\cdot | z) = N(\alpha_t z, \beta_t^2 \mathbf{I}) = \mathcal{N}(0, \mathbf{I})$，这是一个纯高斯分布；在 $t$ 为 $1$，$p_t(\cdot | z) = \mathcal{N}(\alpha_t z, \beta_t^2 \mathbf{I}) = \mathcal{N}(z, 0)$，这是仅仅一个点 $z$。

条件概率路径与流的关系是，给定一个目标数据点 $z$，我们可以设计一个条件流映射（Conditional Flow Mapping） $\phi_t(x_0, z)$。那么，$p_t(x|z)$ 被定义为 $$p_t(x|z) = [\phi_t(\cdot, z)]_* p_0$$这意味着，如果你从 $p_0$ 中采样一个初始点 $x_0$，经过映射 $\phi_t$ 变换后的新变量 $x = \phi_t(x_0, z)$ 所服从的分布，其密度函数就是 $p_t(x|z)$。

同样的，在条件概率路径与条件流之后，我们可以定义条件矢量场。由条件矢量场驱动的流映射，正好能产生我们预设的条件概率路径 $p_t(x|z)$。类似的，$u_t(\cdot|z): \mathbb{R}^d \to \mathbb{R}^d$ 是一个矢量场，或者另一种写法是 $u(\cdot|z): [0, 1] \times \mathbb{R}^d \to \mathbb{R}^d$ 是一个矢量场族。并且矢量场应该满足 $$\frac{d\phi_t(x_0 | z)}{dt} = u_t(\phi_t(x_0 | z) | z)$$

我们用一张动图来解释这一切。在下图中，你可以看到全空间内的点不断向某一点靠近，直至最后变成纯粹的 Dirac 分布。这里演示的就是最优路径下矢量场，因为所有点沿着直线向最终目标移动。

<img src="./assets/ShowCondition.gif" width="400" height="300">

我们继续说损失函数的计算。

实际上条件概率路径仅仅是我们的一种桥梁，我们最终的目的是边缘概率路径，或者说全局概率路径。我们对所有数据样本进行边缘化处理 $$p_t(x) = \int p_t(x|z) p_{data}(z) \mathrm{d}z$$该公式定义了全局的概率演化路径。当 $t=1$ 时，$p_1(x)$ 恰好是原始分布 $q$ 的混合高斯近似。

这里比较反直觉，因为很难理解一个点向着多个数据点运动。我们写成离散形式，也许会帮助理解。假设数据集中只有 $n$ 个点 $\{z_1, z_2, \dots, z_n\}$，那么每一个点的概率就是 $\frac{1}{n}$。此时，$p_t(x) = \int p_t(x|z) p_{data}(z) \mathrm{d}z$ 就变成了 $$p_t(x) = \frac{1}{n} \sum_{i=1}^n p_t(x | z_i)$$
在 $t$ 为 $1$ 时 $$p_1(x) = \frac{1}{n} \sum_{i=1}^n \delta(x - z_i)$$
所以可以想象数据分布的变化。一开始是高斯分布，逐渐向离散的数据点靠拢，最后所有采样都只会在真实数据点中诞生。

下面展示一张图，非常好地演示了这一过程。其中每种不同颜色点代表不同阶段概率分布。对于单一条件概率路径，最终会集中到一点；对于混合的路径，则会在各个数据点处集中。

<img src="./assets/ShowMargin.png" width="550" height="330">

更进一步，我们还可以定义全局边缘矢量场，使得其在给定边缘概率路径下，可以生成这条路径。我们直接给出这个结论$$u_t(x) = \int u_t(x|z) \frac{p_t(x|z)p_{data}(z)}{p_t(x)} \mathrm{d}z$$
注意分数部分 $\frac{p_t(x|z)q(z)}{p_t(x)}$。根据 Bayes 定理，这实际上是后验概率 $p_{data}(z | x_t = x)$。

因此，边缘矢量场在点 $x$ 处的值，实际上是所有可能通往不同样本 $z$ 的条件矢量场的条件期望 $$u_t(x) = \mathbb{E}_{z \sim p_{data}(z | x_t=x)} [u_t(x | z)]$$

关于边缘矢量场为什么是这个形式，我们在附录中详细证明这一点。我们在这里先给出直觉性的解释。实际上在给定条件概率路径情形下，矢量场做的就是统计每个数据点 $x$ 处流量情况。想象一群粒子根据各自的条件概率路径经过数据点 $x$，各自目标 $z$ 各不相同，如果想要给出一个总流速方向，就是取已经在点 $x$ 的粒子的矢量走向取加权平均，这实际上就是 $\mathbb{E}_{z \sim p_{data}(z | x_t=x)} [u_t(x | z)]$。

我们写一个$\textcolor{red}{错误}$的内容，$\textcolor{red}{u_t(x) = \mathbb{E}_{z \sim p_{data}(z)} [u_t(x | z)]}$。这看起来很像全概率公式，实际上完全不对，因为概率场作为标量场和矢量场的可加性是不一样的。Peter Holderrieth 说，建立 Flow Matching 的正确数学直觉很具技巧性，这是正确的。无关难度，实际上取决于理解的方式，建立正确的直觉需要时间，甚至很折磨。

所以，如果对于每一个 $z$，条件矢量场 $u_t(\cdot|z)$ 都能生成对应的条件路径 $p_t(\cdot|z)$，那么构造出的边缘矢量场 $u_t(x)$ 必然能生成边缘概率路径 $p_t(x)$。

不过我们还是难以估计边缘矢量场 $u_t$，原因是真实数据分布 $p_{data}$ 是未知量。注意我们这里说的是难以估计而不是无法估计，原因是根据现在的 $u_t$ 形式，实际上对于每个数据空间中点 $x$，我们都可以通过 Monte Carlo 采样估计此处的 $u_t(x)$。然而这样效率实在是太低下，因为我们计算每个点都需要对全数据样本采样一遍 (或者抽样估计，这是后话的优化)。

原论文作者给出一种巧妙的估计方法。我为你演示。

### 目标函数的转化

我们定义 Conitional Flow Matching 损失函数为$$\mathcal{L}_{\text{CFM}}(\theta) = \mathbb{E}_{t \sim \mathcal{U}[0,1], z \sim p_{data}(z), x \sim p_t(x|z)} \|v_t(x; \theta) - u_t(x|z)\|^2$$
或者有时更喜欢 $$\mathcal{L}_{\text{CFM}}(\theta) = \mathbb{E}_{t \sim \mathcal{U}[0,1], x \sim p_t(x), {z \sim p_t(z|x)}} \|v_t(x; \theta) - u_t(x|z)\|^2$$

一个结论是，$\mathcal{L}_{\text{FM}}(\theta)$ 与我们定义的 $\mathcal{L}_{\text{CFM}}(\theta)$ 关于参数 $\theta$ 的梯度是完全等价的，因此他们作为训练目标函数等价。

回顾一下 $\mathcal{L}_{FM}(\theta)$ $$\mathcal{L}_{\text{FM}}(\theta) = \mathbb{E}_{t \sim \mathcal{U}[0,1], x \sim p_t(x)} \|v_t(x; \theta) - u_t(x)\|^2$$

为你证明 $$\nabla_\theta \mathcal{L}_{\text{FM}}(\theta) = \nabla_\theta \mathcal{L}_{\text{CFM}}(\theta)$$

我们展开 $\mathcal{L}_{\text{FM}}$ 的期望项，略去关于 $t$ 的积分 $$\mathcal{L}_{\text{FM}}(\theta,t) = \int p_t(x) \|v_\theta(x, t) - u_t(x)\|^2 \mathrm{d}x$$
展开平方项并只保留包含 $\theta$ 的项，$u_t(x)^2$ 与 $\theta$ 无关，梯度为 0$$\nabla_\theta \mathcal{L}_{\text{FM}}(\theta,t) = \nabla_\theta \int p_t(x) \left[ v_\theta(x, t)^2 - 2 v_\theta(x, t) \cdot \mathbf{u_t(x)} \right] \mathrm{d}x$$
代入 $u_t(x)$ 的定义$$\int p_t(x) v_\theta(x, t) \cdot \left[ \int u_t(x|z) \frac{p_t(x|z)p_{data}(z)}{\mathbf{p_t(x)}} \mathrm{d}z \right] \mathrm{d}x$$
$p_t(x)$ 在积分项内外抵消了， 剩下的项重新整理为 $$\int \int v_\theta(x, t) \cdot u_t(x|z) p_t(x|z) p_{data}(z) \mathrm{d}z \mathrm{d}x$$
这正好是 $\mathcal{L}_{\text{CFM}}$ 展开后的交叉项 $$\mathcal{L}_{\text{CFM}}(\theta,t) = \mathbb{E}_{z \sim p_{data}, x \sim p_t(\cdot|z)} \|v_\theta(x, t) - u_t(x|z)\|^2$$
注意我们这里其实全部忽略对 $t$ 的均匀采样了，因为对于每个时间步需要优化的目标是关于时间步独立的。所以最终计算损失只需要对时间步取平均即可。

我们证毕了。美丽的飞跃。为什么 $\mathcal{L}_{CFM}(\theta)$ 大大减少了估计的工作量？观察 $\mathcal{L}_{FM}(\theta)$ 与 $\mathcal{L}_{CFM}(\theta)$ 的形式会发现，计算 $\mathcal{L}_{FM}(\theta)$ 需要经历三次 Monte Carlo 估计，计算 $\mathcal{L}_{CFM}(\theta)$ 却仅仅只需要进行一次 Monte Carlo 估计。如果你还没有意识到这件事，我为你演示。

假设当前 Batch 中有 $n$ 个目标数据点 $\{z_1, z_2, \dots, z_n\}$，且已经为每个点采样了对应的初始噪声 $x_{0,i}$ 和时间 $t_i$。设定传输路径为最优路径，即条件位置为 $x_i = (1-t_i)x_{0,i} + t_i z_i$，条件矢量场为 $u(x_i|z_i) = z_i - x_{0,i}$ (我们随后会正式说明这一点)。计算一次 $\mathcal{L}_{FM}(\theta)$

第一次 Monte Carlo 采样是为了估计全局概率路径 $$p_t(x_i) \approx \frac{1}{n} \sum_{j=1}^n p_t(x_i | z_j)$$
为了得到这一个点的密度，需要计算 $n$ 个高斯核函数。整个 Batch 需要计算 $n \times n$ 次核函数，时间复杂度是 $\mathcal{O}(n^2)$。

第二次 Monte Carlo 采样是为了估计全局矢量场 $$u_t(x_i) \approx \frac{\sum_{j=1}^n p_t(x_i | z_j) u_t(x_i | z_j)}{p_t(x_i)}$$ 分子需要计算 $n$ 个矢量与标量的乘积并求和，分母是第一次算好的 $p_t(x_i)$。整个 Batch 同样需要 $O(n^2)$ 的计算量。

第三次 Monte Carlo 采样时为了计算最终损失 $$\mathcal{L} = \frac{1}{n} \sum_{i=1}^n \| v_t(x_i; \theta) - u_t(x_i) \|^2$$
这一步时间复杂度是 $\mathcal{O}(n)$ 级别的，因为对整个 Batch 取平均即可。

所以总计算量时间复杂度是 $\mathcal{O}(n^2)$，做了三次 Monte Carlo 采样。

我们来看 $\mathcal{L}_{CFM}(\theta)$ 计算量。由于我们知道条件概率路径和条件矢量场，实际上可以直接计算损失 $$\mathcal{L} = \frac{1}{n} \sum_{i=1}^n \| v_t(x_i; \theta) - u_t(x_i | z_i) \|^2$$

结束了。对于 $\mathcal{L}_{CFM}(\theta)$ 计算整个 Batch 的损失复杂度仅为 $\mathcal{O}(n)$，Monte Carlo 采样一次。数学的魔法。不过要解释根本缘由的话，可以认为原本 $\mathcal{L}_{FM}$ 是采样完整数据样本后计算损失并且更新一次，极其精确但是缓慢，但是 $\mathcal{L}_{CFM}$ 则指出可以对多个单样本采样计算后计算平均损失再更新一次，我们只是指出了允许更新的更小单位。

### Gaussian 条件路径

我们刚刚说了很久，默认的条件概率路径是高斯路径。现在我们正式给出一个说明，包括对于高斯路径下条件矢量场的计算。

高斯条件路径 $p_t(x|z)$ 是一个随时间步变化的高斯分布 $$p_t(x|z) = \mathcal{N}(x \,|\, \mu_t(z), \sigma_t(z)^2 \mathbf{I})$$
并且需要满足 $$p_0(x|z) = p_0(x) = \mathcal{N}(x \,|\, 0, \mathbf{I})$$在初始时刻，所有条件路径必须坍缩回相同的噪声分布，且与目标样本 $z$ 无关。此时 $\mu_0(z) = 0, \sigma_0(z) = 1$。

而在终止时刻 $$ p_1(x|z) = \delta(x - z) $$ 概率质量必须高度集中在数据点 $z$ 处。此时 $\mu_1(z) = z$ 以及一个极小的 $\sigma_1(z) = \sigma_{\min}$。实际中不直接设置 $\sigma_1(z)$ 为 $0$ 是因为考虑到除零错误。

根据对 $\mu_t$ 和 $\sigma_t$ 的不同设计，高斯条件路径可以衍生出不同的流特性。比如最优传输路径 (Optimal Transport Path)
$$\mu_t(z) = t z \quad \sigma_t = 1 - t$$
在这种设计下，对应的条件矢量场 $u_t(x|z)$ 具有形式 $$u_t(x|z) = \frac{z - x}{1 - t}$$

简单验证一下这件事。建立对应的流映射 $$\psi_t(x_0, z) = \sigma_t x_0 + \mu_t$$ $$\psi_t(x_0, z) = (1 - t) x_0 + t z$$
那么 $$\frac{\partial}{\partial t} \psi_t(x_0, z) = - x_0 + z$$
因为 $$x_0 = \frac{x - t z}{1 - t}$$
得到 $$u_t(x|z) = -\left( \frac{x - t z}{1 - t} \right) + z$$
最后就是 $$u_t(x|z) = \frac{z - x}{1 - t}$$

很多算法喜欢把 $x_0$ 标记为  $\epsilon$，其中 $\epsilon \sim \mathcal{N}(0,\mathbf{I})$，那么 $$x_t = \psi_t(x_0, z) = (1 - t) x_0 + t z = (1 - t) \epsilon + t z \quad \epsilon \sim \mathcal{N}(0,\mathbf{I})$$
所以上面条件矢量场表达式还可以进一步化简 $$u_t(x_t|z) = \frac{z - x_t}{1 - t} = \frac{z - [(1 - t) \epsilon + t z]}{1 - t} = z - \epsilon$$


不过我们这里限制了高斯路径是最优传输路径。更广泛的形式是 $$u_t(x_t|z) = \dot{\sigma_t} \epsilon + \dot{\mu_t} z$$
后续我们讲 VP 或 VE 下传输路径时会使用这个一般形式。或者提前一说。

对于 Variance Preserving，有 $\mu_t(z) = \alpha_t z$ 和 $\sigma_t = \sqrt{1 - \alpha_t^2}$，路径是 $p_t(x|z) = \mathcal{N}(x \,|\, \alpha_t z, (1 - \alpha_t^2) \mathbf{I})$，其中 $\alpha_t$ 从 $0$ 到 $1$。

此时 $$u_t(x|z) = \frac{\dot{\alpha}_t}{1 - \alpha_t^2} (z - \alpha_t x) \quad \text{}$$

对于 Variance Exploding，有 $\mu_t(z) = z$ 和 $\sigma_t = \sigma(t)$，路径是 $p_t(x|z) = \mathcal{N}(x \,|\, z, \sigma_t^2 \mathbf{I})$，其中 $\sigma(t)$ 从极小值演化到极大值。

此时 $$u_t(x|z) = \frac{\dot{\sigma}_t}{\sigma_t} (x - z) \quad \text{}$$

### 训练算法

铺垫已久。我们快速给出训练算法。我们拥有神经网络 $u_t^\theta$ 时待学习的矢量场参数化模型，其输入为位置 $x$ 和时间 $t$，输出为预测的速度矢量。对于一个数据集 $z \sim p_{data}$，首先采样真实数据点 $z$，再从 $[0, 1]$ 范围内均匀采样一个时间 $t$。这确保了神经网络能够学习到从初始噪声到最终数据整个演化过程中的每一个切面的速度信息。

下一步是根据预设的条件概率路径，计算粒子在 $t$ 时刻应该处于的位置 $x$。对于一般 Gaussian 条件路径，这一步通常是 $x = \sigma_t x_0 + \mu_t(z)$，其中 $x_0 \sim \mathcal{N}(0, \mathbf{I})$。所以其实是先采样噪声 $x_0 \sim \mathcal{N}(0, \mathbf{I})$，再根据预设的 $\sigma_t$ 和 $\mu_t$ 决定 $x$。

最后计算 $\mathcal{L}(\theta) = \|u_t^\theta(x) - u_t^{\text{target}}(x|z)\|^2$，其中 $u_t(x_t|z) = \dot{\sigma_t} x_0 + \dot{\mu_t} z$。如果我们做一个 Batch 上的训练，则在对整个 Batch 中数据点重复上述步骤并且对最终损失求平均，最终计算梯度更新参数 $\theta$。

更多的 $$L_{\text{CFM}}(\theta)$$ $$= \mathbb{E}_{t \sim \text{Unif}, z \sim p_{\text{data}}, x \sim p_t(\cdot | z)} \left[ \| u_t^{\theta}(x) - u_t^{\text{target}}(x | z) \|^2 \right]$$ $$= \mathbb{E}_{t \sim \text{Unif}, z \sim p_{\text{data}}, \epsilon \sim \mathcal{N}(0, I_d)} \left[ \| u_t^{\theta}(\mu_t z + \sigma_t \epsilon) - u_t^{\text{target}}(\mu_t z + \sigma_t \epsilon | z) \|^2 \right]$$ $$= \mathbb{E}_{t \sim \text{Unif}, z \sim p_{\text{data}}, \epsilon \sim \mathcal{N}(0, I_d)} \left[ \| u_t^{\theta}(\mu_t z + \sigma_t \epsilon) - (\dot{\mu}_t z + \dot{\sigma}_t \epsilon) \|^2 \right]$$

结束了。如你所见，Flow Matching 的训练算法简洁到了极致，没有 DDIM 复杂的噪声调度，也没有近似的反向 Markovian 过程。

# 推理与采样

Flow Matching 推理的本质是在已知神经网络拟合的矢量场 $u_t^\theta$ 的情况下解下面微分方程给出的轨迹 $$\frac{\mathrm{d}X_t}{\mathrm{d}t} = u_t^\theta(X_t), \quad X_0 \sim p_{\text{init}} \quad$$
一个常用方法是 Euler 法。

我们设定初始时间步 $t$，并且设定步长 $h = \frac{1}{n}$，其中 $n$ 是总采样步数。步数越多，轨迹越逼近真实 ODE，但计算成本越高。

从标准高斯分布 $\mathcal{N}(0, \mathbf{I})$）中随机抽取一个噪声点 $X_0$ 作为生成的起点。

对于之后每个时间步 $t$ 做这些事。将当前位置 $X_t$ 和时刻 $t$ 输入网络 $u_t^\theta$，获取该点处的速度矢量。随后更新$$X_{t+h} = X_t + h \cdot u_t^\theta(X_t) \quad$$
并将时间推进到下一时刻 $t \leftarrow t + h$。

重复上述步骤直至 $t$ 为 $1$。循环结束时，粒子到达 $t = 1$ 时刻的位置 $X_1$。这个点即为最终生成的模拟数据样本。

若训练时采用最优传输路径，神经网络给出的矢量场会倾向于产生直线轨迹。在直线上，Euler 法的离散化误差理论上降为零。这意味着可以使用极大的步长就能获得极高质量的图像生成效果。

实际上也可以将 Euler 法替换成其他 ODE 求解器，比如 RK4。这是优化方向之一，但是 Euler 法已经很强大了。

# FM 与 Diffusion Model

Flow Matching 模型学习的是从高斯分布到真实分布概率转移的矢量场，Diffusion 模型学习的却是得分，或者说概率分布的梯度。然而我们可以观察到，他们的推理过程极其相似。从一个简单的先验分布 $X_0 \sim p_{\text{init}}$ 采样，在 $t \in [0, 1]$ 连续时间内，通过微分方程更新粒子位置，最终返回 $t=1$ 时刻的点 $X_1$ 作为生成样本。但是为什么他们学习了截然不同的场？或者说，其实他们学习的内容是类似的？

我们在这里设置一个疑问：假如我们训练了一个 FM 模型却用 SDE 的推理法（Langevin Dynamics），那么这是一个 Diffuison 模型吗？

答案是，某种意义上是的，但是不完全合法，取决于训练时的传输路径设置。如果训练时用最优传输路径，FM 学习的矢量场是直线。强行加 Langevin 噪声反而会破坏这种直线几何，导致推理效率下降且没有任何增益。然而，如果训练时用的是 VP/VE 路径，此时 FM 学习到的矢量场 $u_t$ 与扩散模型的 Score 直接挂钩，用 SDE 采样是完全合法且数学自洽的。

实际上，对于任何一个 SDE（Diffusion 模型推理时使用的方程），都存在一个对应的概率流 ODE，它们的边缘分布 $p_t(x)$ 在每一时刻都完全一致。这个 ODE 的速度场 $u_t$ 定义为 $$u_t(x) = f(t)x - \frac{1}{2}g^2(t) \nabla \log p_t(x)$$因此如果一个 FM 模型得到了 $u_t$，那么实际上间接拥有了那个对应的得分。

对比一下 FM 与 Diffusion 要解决的方程 $$dX_t = u_t^\theta(X_t)dt$$ $$dX_t = u_t^\theta(X_t)dt + \sigma_t dW_t$$

Flow Matching 是一个更大的框架，它允许定义任何从噪声到数据的移动方式。DDIM 选择了一条遵循高斯加噪规律的路径，也就是 VP，这条路径扭曲且奇怪。如果选择了最笔直的最优路径，效率提升是理所应当的。因此，目前工业界的生成图式 State-of-the-Art 模型全面转向了 Flow Matching。如果你还没发现，我为你演示如何用 FM 的语言叙述 DDIM。

VP 中，样本 $x$ 与噪声 $\epsilon$ 和原始数据 $z$ 的关系为 $$x = \alpha_t z + \sigma_t \epsilon \quad \text{其中 } \sigma_t = \sqrt{1 - \alpha_t^2}$$
由 $u_t(x|z) = \dot{\alpha}_t z + \dot{\sigma}_t \epsilon$，得到 $$u_t(x|z) = \frac{\dot{\alpha}_t}{\alpha_t} x - \frac{\dot{\alpha}_t}{\sigma_t \alpha_t} \epsilon$$

更多的，Flow Matching 学习的全局矢量场 $u_t(\mathbf{x})$, 扩散模型学习的得分场 $\nabla \log p_t(\mathbf{x})$ 以及噪声预测场 $\epsilon_\theta(\mathbf{x}, t)$ 之间存在直接的统一等式 $$u_t(\mathbf{x}) = \frac{\dot{\alpha}_t}{\alpha_t} \left( \mathbf{x} + \nabla \log p_t(\mathbf{x}) \right) = \frac{\dot{\alpha}_t}{\alpha_t} \mathbf{x} - \frac{\dot{\alpha}_t}{\alpha_t \sqrt{1-\alpha_t^2}} \epsilon_\theta(\mathbf{x}, t)$$
所以在获得样本 $x$ 和时间步 $t$ 的情况下预测 $u_t(x|z)$ 等价预测噪声 $\epsilon$，也等价预测得分场 $\nabla \log p_t(\mathbf{x})$。

更有力的证据是，无论是 FM, SMLD 还是 DDIM，神经网络学习的都是一个 $\mathbb{R}^d \to \mathbb{R}^d$ 的矢量场，区别只是矢量场, 得分场与噪声场。

但是这里仅仅是思想实验，请不要直接用未经后训练调整的 FM 模型去做 Diffusion 的推理，或者反过来。原因是许多细节上他们存在区别，比如 FM 接收的时间步是 $t \sim \mathcal{U}[0,1]$，Diffusion 接收的时间步却是 $t \in \{1, 2, \dots, T\}$ (虽然这个细节可以转化统一)。

所以可以看到我们的框架升级历程，从 Score Matching 到 Scored-based Model 到 Flow Matching。

# 总结

本章我们介绍了 Flow Matching，目前最具统治力的生成图式模型训练与推理算法。至此我们应该已经讲解完毕了生成图领域的几大奠基石与里程碑。

本章不知不觉写了很长，原因是 Flow Matching 具备简洁形式的同时背后隐藏了复杂的理论。我认为理解来龙去脉很重要。实际上 FM 的成果相对于 DDIM 与 Scored-based Model 难以阅读很多，需要时间理解。下一章我们来讲述 Diffusion Transformer，也就是我们内核的神经网络。